# WP 09 · Reading & Writing Video
*Computer-vision I/O — Unit II: UAV Data Collection & Processing Methods*

A video is just a loop over images: every image technique in this unit applies per frame. Always `release()` the capture and writer.

**Supporting data:** `flight.mp4` (download it from the button next to this snippet in the playground, then upload when the notebook asks).

---
**How to use:** `Runtime ▸ Run all`, or run each cell top to bottom. The original teaching snippet is reproduced verbatim below; only GUI-only calls (`cv2.imshow`, `cv2.waitKey`, `cv2.destroyAllWindows`) are adapted, because Colab has no display window.

In [ ]:
# === Setup (run me first) ===================================================
# OpenCV, NumPy and Matplotlib are already installed in Google Colab.
# If you run locally and cv2 is missing, uncomment the next line:
# !pip install opencv-python-headless matplotlib

import cv2, numpy as np, os
import matplotlib.pyplot as plt
print("OpenCV", cv2.__version__)

def show(*imgs, titles=None, cmap=None, figsize=(13, 5)):
    """Display 1..N images inline. BGR images are auto-converted to RGB.
    (Colab has no window server, so cv2.imshow() cannot be used.)"""
    titles = titles or [""] * len(imgs)
    plt.figure(figsize=figsize)
    for i, im in enumerate(imgs):
        ax = plt.subplot(1, len(imgs), i + 1)
        if im.ndim == 2:
            ax.imshow(im, cmap=cmap or "gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(titles[i], fontsize=11); ax.axis("off")
    plt.tight_layout(); plt.show()


In [ ]:
# --- Stand-in scene generator (used only if you don't upload the data file) ---
def make_scene(w=768, h=512, seed=7):
    rng = np.random.default_rng(seed)
    img = np.full((h, w, 3), (78, 120, 96), np.uint8)
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.clip(img + (16*np.sin(xx/130) + 10*np.cos(yy/90))[..., None], 0, 255).astype(np.uint8)
    crop = img.copy()
    for k in range(-h, w, 9):
        cv2.line(crop, (k, 0), (k + h, h), (60, 165, 90), 2, cv2.LINE_AA)
    m = np.zeros((h, w), np.uint8); cv2.rectangle(m, (0, 150), (470, h), 255, -1)
    img[m == 255] = crop[m == 255]
    cv2.ellipse(img, (628, 120), (95, 62), 18, 0, 360, (150, 92, 40), -1, cv2.LINE_AA)
    pts = np.array([[0,470],[180,430],[330,360],[430,250],[520,170],[640,90],[w,40]], np.int32)
    cv2.polylines(img, [pts], False, (70, 72, 78), 26, cv2.LINE_AA)
    for cx, cy, bw, bh, col, a in [(150,250,84,60,(205,205,210),8),(250,300,70,52,(120,150,225),-6),
                                   (360,190,60,44,(225,225,230),20),(120,360,56,40,(150,175,235),4)]:
        r = cv2.boxPoints(((cx,cy),(bw,bh),a)).astype(np.int32)
        cv2.fillConvexPoly(img, r, col, cv2.LINE_AA); cv2.polylines(img, [r], True, (40,40,45), 2, cv2.LINE_AA)
    for cx, cy, rr in [(60,120,16),(95,175,13),(300,110,15),(430,430,18),(500,470,14),(700,300,17),(610,380,13),(250,460,15)]:
        cv2.circle(img, (cx, cy), rr, (40, 95, 45), -1, cv2.LINE_AA)
    for cx, cy, col, a in [(300,380,(250,250,250),-32),(470,205,(60,60,235),40)]:
        r = cv2.boxPoints(((cx,cy),(26,12),a)).astype(np.int32); cv2.fillConvexPoly(img, r, col, cv2.LINE_AA)
    return img

def get_image(name, gray=False):
    """Load `name` if present; else offer a Colab upload; else auto-generate."""
    flag = cv2.IMREAD_UNCHANGED if name.lower().endswith(".png") else cv2.IMREAD_COLOR
    if os.path.exists(name):
        im = cv2.imread(name, flag)
        if im is not None:
            print("Loaded", name)
            return cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) if gray else im
    try:
        from google.colab import files
        print(f"Upload '{name}' from the data pack (or press Cancel to auto-generate).")
        up = files.upload()
        for fn in up:
            im = cv2.imread(fn, flag)
            if im is not None:
                print("Loaded", fn)
                return cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) if gray else im
    except Exception:
        pass
    print("Auto-generating a stand-in scene.")
    s = make_scene()
    return cv2.cvtColor(s, cv2.COLOR_BGR2GRAY) if gray else s


In [ ]:
# --- Get a flight video: upload flight.mp4, else synthesise a short clip -----
def get_video(name="flight.mp4"):
    if os.path.exists(name):
        print("Loaded", name); return name
    try:
        from google.colab import files
        print(f"Upload '{name}' from the data pack (or Cancel to auto-generate).")
        up = files.upload()
        for fn in up:
            if fn.lower().endswith((".mp4", ".avi", ".mov")):
                print("Loaded", fn); return fn
    except Exception:
        pass
    print("Auto-generating a short flyover clip...")
    scene = make_scene(); H0, W0 = scene.shape[:2]
    big = cv2.resize(scene, (int(W0*1.7), int(H0*1.7)))
    OW, OH, FPS, N = 480, 320, 24, 96
    vw = cv2.VideoWriter(name, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (OW, OH))
    for i in range(N):
        t = i/(N-1); te = 0.5-0.5*np.cos(np.pi*t)
        x = int((big.shape[1]-OW)*te); y = int((big.shape[0]-OH)*te)
        vw.write(big[y:y+OH, x:x+OW].copy())
    vw.release(); return name


### The snippet, exactly as shown in the playground
```python
cap = cv2.VideoCapture("flight.mp4")   # or 0 = webcam / FPV feed

fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("annotated.mp4", fourcc, fps, (w, h))

while True:
    ret, frame = cap.read()   # ret=False at end of file
    if not ret:
        break
    # ... process frame (any lab from WP 10-15) ...
    out.write(frame)
    if cv2.waitKey(1) == ord("q"):
        break

cap.release(); out.release(); cv2.destroyAllWindows()
```

In [ ]:
# --- Runnable version: annotate every frame, then play it back inline --------
src = get_video("flight.mp4")
cap = cv2.VideoCapture(src)

fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"fps={fps:.1f}  size={w}x{h}")

# Colab-friendly writer: H.264 in an .mp4 so it plays in the browser.
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("annotated.mp4", fourcc, fps, (w, h))

n = 0
while True:
    ret, frame = cap.read()          # ret=False at end of file
    if not ret:
        break
    # ---- process frame: a simple Canny-edge overlay (a WP-12 lab) ----
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 60, 140)
    frame[edges > 0] = (0, 0, 255)   # paint edges red
    cv2.putText(frame, f"frame {n}", (10, h - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    out.write(frame)
    # (In a desktop app you'd cv2.imshow here and break on 'q'. Colab can't.)
    n += 1

cap.release(); out.release()
print(f"Wrote annotated.mp4 ({n} frames)")

In [ ]:
# --- Play the annotated video inline ---------------------------------------
# Re-encode to H.264/yuv420p so the browser can play it, then embed.
import subprocess, base64
from IPython.display import HTML
subprocess.run(["ffmpeg","-y","-loglevel","error","-i","annotated.mp4",
                "-c:v","libx264","-pix_fmt","yuv420p","-movflags","+faststart",
                "annotated_h264.mp4"], check=False)
path = "annotated_h264.mp4" if os.path.exists("annotated_h264.mp4") else "annotated.mp4"
b64 = base64.b64encode(open(path, "rb").read()).decode()
HTML(f'<video width=480 controls src="data:video/mp4;base64,{b64}"></video>')